# Импорты

## Параметры синхронизации

In [ ]:
!pip install -q papermill

import papermill as pm
import pandas as pd
import gdown
import os

In [ ]:
if 'input_filename' not in locals():
    input_filename = '/content/data_1_2.xlsx' # имя файла по умолчанию для ручного запуска
    print(f"Запуск вручную. Используем файл: {input_filename}")
else:
    print(f"Пайплайн запущен! Обрабатываем файл: {input_filename}")

if os.path.exists(input_filename):
    df = pd.read_excel(input_filename)
    print("Файл успешно загружен в DataFrame")
else:
    print(f"Ошибка! Файла {input_filename} нет в папке /content/")

## Остальные инпуты

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy import stats
import re
from datetime import datetime

In [ ]:
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
#df = pd.read_excel(input_filename)
df.head(3)

In [ ]:
df.info()

# Предобработка

### Сдвиг колонок

В изначально представленном датасете поехали строчки, поэтому так можно было это легко отфильтровать

In [ ]:
cols = df.columns
cols

In [ ]:
df['Регион'].value_counts()

In [ ]:
obj_cols = ['shift_org_1', 'shift_reg_1', 'shift_spec_1']

df[obj_cols] = df[obj_cols].apply(
    pd.to_numeric, errors='coerce'
)

In [ ]:
df.condition_org.value_counts()[:15]

In [ ]:
bad_condition_org = ['0000']

df['condition_org'] = df['condition_org'].where(
    df['condition_org'].apply(lambda x: isinstance(x, str)),
    None
)

df.loc[df['condition_org'].isin(bad_condition_org), 'condition_org'] = np.nan

df.condition_org.value_counts()[:15]

,count
condition_org,
В амбулаторных условиях,4679
В стационарных условиях,4328
В организации смешанного типа,790
Скорая помощь и медицина катастроф,696
В условиях дневного стационара,324
Вне медицинской организации,259


In [ ]:
df = df.dropna(subset=['condition_org'])

### Удаление пустых колонок

In [ ]:
df = df.drop(columns='For FA')
df.head(1)

In [ ]:
df = df.drop(columns='type_contract')

In [ ]:
df = df.drop(columns='Organisation')

### Медицинская организация

In [ ]:
df.employment.unique()

array(['Врач', 'Врач, не работаю с пациентами напрямую',
       'Средний медицинский персонал',
       'Средний медицинский персонал, не работаю с пациентами напрямую'],
      dtype=object)

In [ ]:
def rename_emp(x):
  if x == 'Средний медицинский персонал':
    return 'Средний МП'
  elif x == 'Средний медицинский персонал, не работаю с пациентами напрямую':
    return 'Средний МП, не раб. с пациентами напрямую'
  elif x == 'Врач, не работаю с пациентами напрямую':
    return 'Врач, не раб. с пациентами напрямую'
  else:
    return x


df["employment"]  = df["employment"].apply(rename_emp)
df["employment2"]  = df["employment2"].apply(rename_emp)
df.employment.value_counts()

### Опыт работы 😥😥😥

In [ ]:
df['exper_org'].nunique()

650

In [ ]:
df.exper_org.head()

,exper_org
0,33
1,20
2,16
3,24
4,9


In [ ]:
df['exper_org'] = df['exper_org'].replace(['nan', 'NaN'], '-1')

In [ ]:
repl_years = ["года", "лет", "год", "г", "г.", "полтора года"]
trash_text = ["более", "около", "мед сестра", ")", "профилакторий", ">", "менее", "много"]

def clean_years(x):
  x = str(x).lower()
  x = x.strip()

  for _ in trash_text: #текст не меняющий смысл
    x = x.replace(_, "").strip()

  if not x.isnumeric():

    if x in repl_years:
      return 1

    for word in repl_years:
      x_clean = x.replace(word, "").strip()#формат 2 года
      if x_clean.isnumeric():
        return x_clean

    if x in ['нет', 'нету', 'студент', 'месяц',  'месяца', "меньше года", "менее года", "да", "-", "...", "не", "пол года", "<1"]:
      return 0

    if len(x) == 0 or len(x) > 15:
      return 0

    if x == 'з':
      return 3

    x_split = x.split()
    if x_split[0].isnumeric():
      if len(x_split) == 6 and x_split[1] in ['л', 'г']:  #5 л 2 м 0 дн формат
        print(x)
        return x_split[0]

      elif len(x_split) == 2 and x_split[1][:3] == "мес" and int(x_split[0]) < 12: #формат 2 месяца
        return 0

      elif x_split[1][:3] in ["год", "лет"]:
        return x[0]



    if x[0].isnumeric() and (x[1:4] in ["мес", "нед"] or x[1] == "м"): #6месяцев
      return 0
    if x[:1].isnumeric() and x[2:5] in ["мес", "нед"]: #6месяцев
      return 0
    elif x[0].isnumeric() and (x[1:4] == "год" or x[1] == "г"): #1год, 4года и 2 месяца
      return x[0]

    #разбираюсь с нецелыми годами 3.5, 11.5
    for i in [3, 4]:
      if bool(re.fullmatch( r'\d+,\d+', x[:i])):
        return x[:i].split(",")[0]
      if bool(re.fullmatch( r'\d+.\d+', x[:i])):
        return x[:i].split(".")[0]

    match = re.search(r'\d+', x)
    if match:
        return int(match.group())

  return x

df["exper_org"] = df["exper_org"].apply(clean_years)

5 л 2 м 0 дн
31 г 4 м 8 дн
0 л 1 м 15 дн
28 л 2 м 1 дн
28 л 10 м 14 дн
2 г 5 м 3 дн
2 г 5 м 3 дн


In [ ]:
df[~pd.to_numeric(df["exper_org"], errors='coerce').notna()].dropna(subset=["exper_org"])["exper_org"]

,exper_org
1270,ссмп
1308,приличный
1407,достаточный
1527,гауз ккб смп
1577,стом. пол.
...,...
12245,nan
12342,nan
12455,nan
12498,nan


In [ ]:
df["exper"] = df["exper"].apply(clean_years)
df["exper_reg"] = df["exper_reg"].apply(clean_years)
df["exper_org"] = df["exper_org"].apply(clean_years)

5 л 2 м 0 дн
31 г 4 м 8 дн
0 л 1 м 15 дн
28 л 2 м 1 дн
31 г 4 м 9 дн
2 г 5 м 3 дн
2 г 5 м 3 дн


In [ ]:
non_numeric = df[pd.to_numeric(df["exper"], errors="coerce").isna()]
print(non_numeric["exper"].unique())

['nan' '.' '_']


In [ ]:
non_numeric = df[pd.to_numeric(df["exper_reg"], errors="coerce").isna()]
print(non_numeric["exper_reg"].unique())

df["exper_reg"] = pd.to_numeric(df["exper_reg"], errors="coerce").fillna(0)
non_numeric = df[pd.to_numeric(df["exper_reg"], errors="coerce").isna()]
print(non_numeric["exper_reg"].unique())

['мариинск' 'ментше года' 'большоц' 'сорок два года' 'не помню' 'кемерово'
 'шесть лет' 'зз' 'не знаю' '8л4' '.' '_' 'зо' 'тот же' 'сорок один год'
 'сорокдевять' '33-3' '7-8' '3-4' ',' 'nightcore' 'небольшой' '10-1'
 '31-3' 'два месяца' 'зо лет' 'nan' 'всю жизнь' 'хз' '4 4' '41г6' 'незнаю']
[]


In [ ]:
df[~pd.to_numeric(df["exper_org"], errors='coerce').notna()].dropna(subset=["exper_org"])["exper_org"]

,exper_org
1270,ссмп
1308,приличный
1407,достаточный
1527,гауз ккб смп
1577,стом. пол.
...,...
12245,nan
12342,nan
12455,nan
12498,nan


In [ ]:
df['exper']

,exper
0,12
1,20
2,16
3,24
4,9
...,...
12587,nan
12590,49
12591,30
12593,16


In [ ]:
df["exper"] = pd.to_numeric(df["exper"], errors="coerce").fillna(0)
df["exper_org"] = pd.to_numeric(df["exper_org"], errors="coerce").fillna(0)
df["exper_reg"] = pd.to_numeric(df["exper_reg"], errors="coerce").fillna(0)

In [ ]:
df["exper_org"] = pd.to_numeric(df["exper_org"], errors="coerce")
df["exper_reg"] = pd.to_numeric(df["exper_reg"], errors="coerce")
df["exper"] = pd.to_numeric(df["exper"], errors="coerce")

In [ ]:
df['exper'] = df['exper'].replace(-1, np.nan)
df['exper_org'] = df['exper_org'].replace(-1, np.nan)
df['exper_reg'] = df['exper_reg'].replace(-1, np.nan)

In [ ]:
def fix_experience(df):
    mask = (df['exper_org'] > 80) & ((df['exper_org'] // 100) == df['exper'])

    df.loc[mask, 'exper_org'] = df['exper_org'] // 100

    return df

def fix_experience2(df):
    mask = (df['exper_org'] > 80) & ((df['exper_org'] // 10) == df['exper'])

    df.loc[mask, 'exper_org'] = df['exper_org'] // 10
    return df

df = fix_experience(df)

In [ ]:
import numpy as np

def set_null_experience(df):
    df.loc[df['exper_org'] > 90, 'exper_org'] = 0
    return df

df = set_null_experience(df)

In [ ]:
def fix_experience3(df):
    mask = (df['exper_org'] > df['exper'])

    df.loc[mask, 'exper_org'] = df['exper']

    return df

df = fix_experience3(df)

### avr_work_hours

In [ ]:
df['avr_work_hours'] = df['avr_work_hours'].abs()

In [ ]:
print(df[df['avr_work_hours'].astype(float) < 10]['avr_work_hours'].value_counts())

In [ ]:
print(df[df['avr_work_hours'].astype(float) > 84]['avr_work_hours'].value_counts())

In [ ]:
df.loc[df['avr_work_hours'] > 84, 'avr_work_hours'] /= 4

In [ ]:
print(df[df['avr_work_hours'].astype(float) > 84]['avr_work_hours'].value_counts())

In [ ]:
df.loc[df['avr_work_hours'] > 84, 'avr_work_hours'] = np.nan

In [ ]:
df['avr_work_hours'] = df['avr_work_hours'].fillna(40)

### wage_rate

In [ ]:
df['wage_rate'].value_counts()

In [ ]:
df['wage_rate'] = df['wage_rate'].replace('Более 3', 4)

In [ ]:
df['wage_rate'] = df['wage_rate'].fillna(1)

### org_type

In [ ]:
df['org_type'].value_counts()

In [ ]:
df['org_type'] = df['org_type'].replace('-', np.nan)
df['org_type'] = df['org_type'].fillna('Другие')

### Возраст

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3))

df["age"].plot(kind="hist", bins=20, ax=axes[0])
axes[0].set_title("Age")

plt.tight_layout()
plt.show()

In [ ]:
def get_age(x):
  if x < 14:
    return "0 - Выброс, < 14"
  elif x < 18:
    return "1 - Подростковый, 13-17"
  elif x < 25:
    return "2 - Юный, "
  elif x < 45:
    return "3 - Молодой, 25-44"
  elif x < 60:
    return "4 - Средний, 45-59"
  elif x < 75:
    return "5 - Пожилой, 60-74"
  elif x < 90:
    return "6 - Старческий, 75-89"
  elif x >= 90:
    return "7 - Долгожители, > 90"
  else:
    return "Отсутствует"

df["age_number"]  = df["age"]
df["age"]  = df["age_number"].apply(get_age)

In [ ]:
df[df["age"] == "7 - Долгожители, > 90"].age_number

,age_number
453,94.00
1563,92.00
9654,98.00


In [ ]:
df['age'].value_counts()

In [ ]:
df['age'] = df['age'].replace('Отсутствует', np.nan)

In [ ]:
df['age_number'].mean()

np.float64(44.53321742313324)

In [ ]:
df['age_number'] = df['age_number'].fillna(44)
df['age'] = df['age'].fillna('3 - Молодой, 25-44')

### Пустые значения

In [ ]:
df['qualification'] = df['qualification'].fillna('Нет')

In [ ]:
df['shift_org_1'] = df['shift_org_1'].fillna(5)

In [ ]:
df['shift_reg_1'] = df['shift_reg_1'].fillna(5)

In [ ]:
df['shift_spec_1'] = df['shift_spec_1'].fillna(5)

In [ ]:
df['specialisation'] = df['specialisation'].fillna('не указано')

In [ ]:
#строки с большим количеством пустот

In [ ]:
df.drop(columns=['comments']).isna().agg(["sum", "mean"]).T.rename(columns={"sum": "missing_cnt", "mean": "missing_ratio"}).sort_values(by="missing_cnt", ascending=False).head(20) #смотрим пустые столбцы

,missing_cnt,missing_ratio
income_level,22.00,0.00
expectation_prof,17.00,0.00
academic_degree,16.00,0.00
home_duty,14.00,0.00
change_hours_numb,14.00,0.00
gender,13.00,0.00
schedule,10.00,0.00
Столбец1,0.00,0.00
Столбец2,0.00,0.00
Часы,0.00,0.00


In [ ]:
#смотрю пустые значения в строках
df.drop(columns=['comments']).isna().sum(axis=1).value_counts().sort_index(ascending=False)

,count
6,6
5,1
3,1
2,7
1,48
0,11013


In [ ]:
cols_to_check = df.columns.drop(['comments'])
df = df.dropna(subset=cols_to_check)

# Экспорт файла

In [ ]:
today = str(datetime.now().strftime('%d_%m_%Y'))
output_filename = 'preprocessing_result_' + today + '.xlsx'
print(output_filename)

preprocessing_result_11_05_2026.xlsx


In [ ]:
df.to_excel(output_filename, index=False)

print(f"Файл успешно сохранен как: {output_filename}")